# Exploration of FD001 Dataset

## SECTION 1 — Project Overview

**Predictive Maintenance** utilizes historical sensor data to predict when equipment will fail.
**Remaining Useful Life (RUL)** is the amount of time (in cycles) an engine is expected to operate before failure.
**NASA C-MAPSS** dataset contains simulated turbofan engine degradation data over multiple operational cycles.
**Why FD001?** FD001 is the simplest subset with a single operating condition and a single fault mode (HPC degradation), making it ideal for establishing our data foundation.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sys
import os
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

from src.data.loader import load_subset

## SECTION 2 — Load FD001

In [ ]:
train_df, test_df, test_rul = load_subset("FD001")

## SECTION 3 — Dataset Shape

In [ ]:
print("Train Shape:", train_df.shape)
print("Test Shape:", test_df.shape)
print("Test RUL Shape:", test_rul.shape)

print(f"\nNumber of training engines: {train_df['unit'].nunique()}")
print(f"Number of test engines: {test_df['unit'].nunique()}")
print(f"Number of columns: {len(train_df.columns)}")

## SECTION 4 — Raw Data Inspection

In [ ]:
display(train_df.head())
display(test_df.head())
display(test_rul.head())

In [ ]:
train_df.info()

In [ ]:
test_df.info()

## SECTION 5 — Missing Values

In [ ]:
print("Missing values in Train:", train_df.isnull().sum().sum())
print("Missing values in Test:", test_df.isnull().sum().sum())
print("Missing values in Test RUL:", test_rul.isnull().sum().sum())

## SECTION 6 — Engine Lifetime

In [ ]:
engine_lifetimes = train_df.groupby("unit")["cycle"].agg(min_cycle="min", max_cycle="max", num_observations="count").reset_index()
display(engine_lifetimes.head())
display(engine_lifetimes["max_cycle"].describe())

plt.figure(figsize=(10, 6))
sns.histplot(engine_lifetimes["max_cycle"], bins=20, kde=True)
plt.title("Distribution of Final Training Cycles (Engine Lifetimes)")
plt.xlabel("Max Cycles (Lifetime)")
plt.ylabel("Count")
plt.show()

## SECTION 7 — Feature Statistics

In [ ]:
settings_cols = [c for c in train_df.columns if c.startswith("setting_")]
sensors_cols = [c for c in train_df.columns if c.startswith("sensor_")]

def calculate_stats(df, cols):
    stats = []
    for c in cols:
        stats.append({
            "feature": c,
            "variance": df[c].var(),
            "std_dev": df[c].std(),
            "unique_values": df[c].nunique()
        })
    return pd.DataFrame(stats).sort_values(by="variance", ascending=False)

print("\n--- Operational Settings ---")
display(calculate_stats(train_df, settings_cols))

print("\n--- Sensors ---")
display(calculate_stats(train_df, sensors_cols))

## SECTION 8 — Test RUL Distribution

In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(test_rul["RUL"], bins=20, kde=True)
plt.title("Distribution of True Test RUL Values at Test-Series Cutoff")
plt.xlabel("Remaining Useful Life (Cycles)")
plt.ylabel("Count")
plt.show()

## SECTION 9 — Sensor Trajectories

In [ ]:
unit_id = 1
unit_df = train_df[train_df["unit"] == unit_id]

# Pick a few sensors that typically show degradation to visualize
sensors_to_plot = ["sensor_2", "sensor_3", "sensor_4", "sensor_7"]

fig, axes = plt.subplots(len(sensors_to_plot), 1, figsize=(10, 12), sharex=True)
fig.suptitle(f"Sensor Trajectories for Training Engine Unit {unit_id}", fontsize=16)

for i, sensor in enumerate(sensors_to_plot):
    axes[i].plot(unit_df["cycle"], unit_df[sensor])
    axes[i].set_ylabel(sensor)

axes[-1].set_xlabel("Cycle")
plt.tight_layout()
plt.show()

## SECTION 10 — Basic Observations

**OBSERVATION:** Several sensors have a variance of 0.0 (e.g., sensor_1, sensor_10, sensor_18, sensor_19).
**INTERPRETATION:** These sensors do not change over the entire operational lifetime of any engine in the FD001 dataset.
**DECISION:** Keep all features for now to ensure consistency across all subsets. Feature selection will be done later.

**OBSERVATION:** Missing values are 0 across train, test, and test_rul.
**INTERPRETATION:** Data is well-formatted and clean.
**DECISION:** No imputation is required.

**OBSERVATION:** Engine lifetimes range from 128 to 362 cycles, with an average around 206.
**INTERPRETATION:** Engines fail at different operating times, highlighting the importance of condition-based modeling instead of average-lifetime guessing.
**DECISION:** Use actual condition sensors for sequence modeling in future milestones.

## SECTION — TRAINING RUL LABEL ENGINEERING

**Why raw RUL is calculated this way:**
RUL is defined as the time (in cycles) remaining before an engine fails. For the training data, we observe each engine until failure. Thus, the RUL at any given cycle is simply the engine's final observed cycle minus the current cycle.

**Why the final training cycle has RUL = 0:**
At the final cycle, the engine has failed (or reached the end of its useful life). Therefore, it has 0 cycles of remaining life.

**Why early-life RUL clipping can be useful for supervised learning:**
In the early life of an engine, degradation is typically negligible or unobservable, meaning sensor readings remain relatively constant. A model trying to predict a very high RUL (e.g., 300 cycles) based on early-life data may struggle because the engine looks exactly the same as one with 250 cycles remaining. Clipping the RUL assumes a constant "healthy" state until degradation begins to manifest. This simplifies the learning task.

*Clipping is a modeling decision, not a change to the physical definition of RUL.*
* RUL is retained as the original target.
* RUL_clipped is a separate modeling target.

**Note:** 125 is our initial baseline cap and will be evaluated later.

In [ ]:
from src.data.rul import add_training_targets

train_with_targets = add_training_targets(train_df)
display(train_with_targets.head())

In [ ]:
import src.config as config

print("1. Every training engine reaches RUL = 0:", (train_with_targets.groupby("unit")["RUL"].min() == 0).all())
print("2. Minimum RUL is:", train_with_targets["RUL"].min())
print("3. Maximum RUL is:", train_with_targets["RUL"].max())
print("\n4. Distribution statistics for RUL:\n", train_with_targets["RUL"].describe())
print("\n5. Distribution statistics for RUL_clipped:\n", train_with_targets["RUL_clipped"].describe())

num_clipped = (train_with_targets["RUL"] > config.DEFAULT_RUL_CAP).sum()
pct_clipped = num_clipped / len(train_with_targets) * 100
print(f"\n6. Number of observations affected by clipping: {num_clipped}")
print(f"Percentage affected: {pct_clipped:.2f}%")

In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(train_with_targets["RUL"], bins=30, kde=True)
plt.title("Raw RUL Distribution (Training)")
plt.xlabel("Raw RUL (Cycles)")
plt.ylabel("Count")
plt.show()

plt.figure(figsize=(10, 6))
sns.histplot(train_with_targets["RUL_clipped"], bins=30, kde=True, color="orange")
plt.title(f"Clipped RUL Distribution (Cap={config.DEFAULT_RUL_CAP})")
plt.xlabel("Clipped RUL (Cycles)")
plt.ylabel("Count")
plt.show()

unit_id = 1
unit_targets = train_with_targets[train_with_targets["unit"] == unit_id]

plt.figure(figsize=(10, 6))
plt.plot(unit_targets["cycle"], unit_targets["RUL"], label="Raw RUL", linestyle="--")
plt.plot(unit_targets["cycle"], unit_targets["RUL_clipped"], label="Clipped RUL", linewidth=2)
plt.title(f"RUL Trajectory for Engine {unit_id}")
plt.xlabel("Cycle")
plt.ylabel("Remaining Useful Life")
plt.legend()
plt.grid(True)
plt.show()

## SECTION 11 — Feature Statistics

We analyze the candidate features to identify constant and near-constant features.

In [ ]:
from src.data.features import get_feature_columns, calculate_feature_statistics, find_constant_features, find_near_constant_features

candidate_features = get_feature_columns(train_with_targets)
print(f"Total candidate features: {len(candidate_features)}")

feature_stats = calculate_feature_statistics(train_with_targets, candidate_features)
display(feature_stats.head(10))

exact_constants = find_constant_features(feature_stats)
print(f"Exact constant features (variance == 0): {exact_constants}")

near_constants = find_near_constant_features(feature_stats, variance_threshold=0.01)
print(f"Near-constant candidates (variance <= 0.01): {near_constants}")

## SECTION 12 — Target Correlation

Note: This is exploratory correlation, not final feature selection.

In [ ]:
from src.data.features import calculate_target_correlations

corr_rul = calculate_target_correlations(train_with_targets, candidate_features, "RUL")
corr_rul_clipped = calculate_target_correlations(train_with_targets, candidate_features, "RUL_clipped")

plt.figure(figsize=(12, 6))
sns.barplot(data=corr_rul_clipped, x="correlation", y="feature", orient="h")
plt.title("Exploratory Pearson Correlation with RUL_clipped")
plt.show()

## SECTION 13 — Feature Redundancy

Feature-feature correlation heatmap to identify highly redundant sensors.

In [ ]:
from src.data.features import calculate_feature_correlation_matrix

corr_matrix = calculate_feature_correlation_matrix(train_with_targets, candidate_features)
plt.figure(figsize=(16, 12))
sns.heatmap(corr_matrix, cmap="coolwarm", center=0, annot=False)
plt.title("Feature-Feature Correlation Heatmap")
plt.show()

## SECTION 14 — Feature Selection Decision

**Removed Constant Features:** We remove features with exactly 0 variance. These provide no information to the model.
**Retained Features:** We do NOT remove features solely based on low correlation, as complex non-linear relationships might exist that Pearson correlation misses.
**Operational Settings:** FD001 is a single operating condition, but the settings might capture small variations or noise. Since they are near-constant but not perfectly constant, we will retain them in the baseline and let the model decide their utility.

In [ ]:
from src.data.feature_selection import select_fd001_features

selected_features, removed_constant_features, analysis_summary = select_fd001_features(train_with_targets)
print("Removed exact constant features:", removed_constant_features)
print(f"Selected {len(selected_features)} baseline features.")
print("Selected Features:", selected_features)

## SECTION 15 — Cycle Analysis

**Why cycle may be predictive in C-MAPSS:**
Engine degradation typically unfolds over time; therefore, `cycle` is a strong proxy for age and accumulated wear.

**Why cycle can also be dangerous:**
If included, a model might over-rely on `cycle` to learn the 'average lifetime' of engines rather than relying on the condition sensors. The goal of condition-based maintenance (CBM) is to predict RUL based on physical condition, not just age.

**Baseline Feature Matrix Decision:**
For the baseline model, we will **exclude** `cycle` from the predictive features. We want to evaluate the predictive power of the sensors alone. If we find later that sensor signals are too noisy or ambiguous, we may reintroduce cycle or a time-based feature (like a rolling window). But to prove true condition-based learning, the baseline must rely on sensor states.

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(data=train_with_targets, x="cycle", y="RUL_clipped", alpha=0.1)
plt.title("Cycle vs RUL_clipped")
plt.show()

cycle_corr_rul = train_with_targets["cycle"].corr(train_with_targets["RUL"])
cycle_corr_rul_clipped = train_with_targets["cycle"].corr(train_with_targets["RUL_clipped"])
print(f"Cycle correlation with RUL: {cycle_corr_rul:.3f}")
print(f"Cycle correlation with RUL_clipped: {cycle_corr_rul_clipped:.3f}")

## SECTION 16 — Engine-Level Train/Validation Split

**Why row-level random splitting is inappropriate for C-MAPSS:**
A single engine contains many sequential observations, and observations from the same engine are highly correlated. Randomly splitting individual rows causes data leakage because the model would see data from an engine in training and then predict on that same engine's data in validation. The split must occur at the ENGINE level.

We split complete engine IDs into train and validation groups.

In [ ]:
from src.data.split import split_by_engine

train_split, val_split = split_by_engine(train_with_targets, validation_size=0.2, random_state=42)

print(f"Total unique engines: {train_with_targets['unit'].nunique()}")
print(f"Training engine count: {train_split['unit'].nunique()}")
print(f"Validation engine count: {val_split['unit'].nunique()}")
print(f"Training row count: {len(train_split)}")
print(f"Validation row count: {len(val_split)}")
print(f"Validation proportion (rows): {len(val_split) / len(train_with_targets):.2%}")

print("\nTraining engine IDs:\n", sorted(train_split['unit'].unique()))
print("\nValidation engine IDs:\n", sorted(val_split['unit'].unique()))

## SECTION 17 — Leakage Validation

Explicitly verify that the split invariants hold.

In [ ]:
from src.data.split import validate_engine_split

try:
    validate_engine_split(train_with_targets, train_split, val_split)
    print("Leakage Validation: PASS")
    
    train_engines = set(train_split['unit'])
    val_engines = set(val_split['unit'])
    overlap = train_engines.intersection(val_engines)
    print(f"Train engine IDs ∩ validation engine IDs = {overlap}")
    print(f"Train rows + validation rows = {len(train_split) + len(val_split)} (Original: {len(train_with_targets)})")
    print(f"Every engine appears exactly once: {len(train_engines) + len(val_engines) == train_with_targets['unit'].nunique()}")
except ValueError as e:
    print(f"Leakage Validation: FAIL - {e}")

## SECTION 18 — Regression Dataset Preparation

Prepare the final feature matrices (X) and target vectors (y).

In [ ]:
from src.data.dataset import prepare_regression_data

X_train, y_train, X_validation, y_validation = prepare_regression_data(
    train_split, 
    val_split, 
    target="RUL_clipped"
)

print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"X_validation shape: {X_validation.shape}")
print(f"y_validation shape: {y_validation.shape}")

print("\nX_train columns:\n", X_train.columns.tolist())
print("\nX_validation columns:\n", X_validation.columns.tolist())

identical_columns = (X_train.columns == X_validation.columns).all()
print(f"\nIdentical feature columns: {identical_columns}")
no_meta_cols = all(col not in X_train.columns for col in ['unit', 'cycle', 'RUL', 'RUL_clipped'])
print(f"No unit, cycle, RUL, RUL_clipped in X: {no_meta_cols}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True)
sns.histplot(y_train, bins=30, kde=True, ax=axes[0], color='blue')
axes[0].set_title("y_train Distribution")
axes[0].set_xlabel("Clipped RUL")

sns.histplot(y_validation, bins=30, kde=True, ax=axes[1], color='orange')
axes[1].set_title("y_validation Distribution")
axes[1].set_xlabel("Clipped RUL")

plt.tight_layout()
plt.show()

## SECTION 19 — Leakage Caveat

The FD001 feature-selection milestone was exploratory and used the complete labeled training dataset to establish the initial 17-feature baseline.

For a strictly leakage-free production training pipeline, feature selection/statistical fitting must be performed using training engines only and then applied to validation/test data.

The current milestone establishes the engine-level split and prepares the architecture for that stricter pipeline. DO NOT hide this limitation.

## SECTION 20 — Baseline Modeling Setup

**Why we need a baseline:**
A baseline model provides a benchmark of predictive performance. It allows us to understand how difficult the dataset is and whether complex sequential models (like LSTM or CNN) genuinely add value.

**Why Random Forest is useful:**
It is a powerful, non-linear ensemble algorithm that requires minimal preprocessing (no scaling needed for trees). It handles mixed feature behavior well and naturally extracts feature importances, providing interpretability before we move to black-box deep learning.

**Why XGBoost is useful:**
Gradient Boosting typically outperforms Random Forests on tabular data when tuned. A baseline XGBoost configuration helps establish a competitive performance ceiling for standard machine learning approaches.

**Why we are not tuning yet:**
Our goal is to establish a robust evaluation harness, verify our engine-level splits, and secure an initial metric. Premature optimization causes overfitting. We want a reference point, not the final model.

**Why validation is engine-level:**
As established in Milestone 4, rows from the same engine are highly correlated. An engine-level split ensures no leakage, testing the model's ability to generalize to unseen engines.

In [ ]:
from src.data.dataset import prepare_regression_data

X_train, y_train, X_validation, y_validation = prepare_regression_data(
    train_split, val_split, target="RUL_clipped"
)

print("X_train shape:", X_train.shape)
print("X_validation shape:", X_validation.shape)
print("y_train shape:", y_train.shape)
print("y_validation shape:", y_validation.shape)

## SECTION 21 — Random Forest Baseline

Train a Random Forest model on the baseline features.

In [ ]:
from src.models.baseline import train_random_forest, predict_rul, SKLEARN_AVAILABLE
from src.models.metrics import rmse_score, nasa_phm08_score
from src.models.pipeline import get_prediction_diagnostics

if SKLEARN_AVAILABLE:
    rf_model = train_random_forest(X_train, y_train, random_state=42)
    print("Random Forest Training complete.")
    print("Model Configuration:\n", rf_model)
    
    rf_preds = predict_rul(rf_model, X_validation)
    rf_rmse = rmse_score(y_validation, rf_preds)
    rf_nasa = nasa_phm08_score(y_validation, rf_preds)
    
    print(f"\nValidation RMSE: {rf_rmse:.4f}")
    print(f"Validation NASA PHM08 Score: {rf_nasa:.4f}")
    
    rf_diag = get_prediction_diagnostics(val_split, rf_preds, target="RUL_clipped")
    display(rf_diag.head())
else:
    print("scikit-learn is not available in the current environment.")

## SECTION 22 — XGBoost Baseline

Train an XGBoost model on the baseline features.

In [ ]:
from src.models.baseline import train_xgboost, XGBOOST_AVAILABLE

if XGBOOST_AVAILABLE:
    xgb_model = train_xgboost(X_train, y_train, random_state=42)
    print("XGBoost Training complete.")
    print("Model Configuration:\n", xgb_model)
    
    xgb_preds = predict_rul(xgb_model, X_validation)
    xgb_rmse = rmse_score(y_validation, xgb_preds)
    xgb_nasa = nasa_phm08_score(y_validation, xgb_preds)
    
    print(f"\nValidation RMSE: {xgb_rmse:.4f}")
    print(f"Validation NASA PHM08 Score: {xgb_nasa:.4f}")
    
    xgb_diag = get_prediction_diagnostics(val_split, xgb_preds, target="RUL_clipped")
else:
    print("xgboost is not installed. XGBoost Baseline could not be executed. Continuing with Random Forest limits.")

## SECTION 23 — Model Comparison

Compare the baselines using the pipeline runner.

In [ ]:
from src.models.pipeline import run_baseline_models

results_df = run_baseline_models(train_split, val_split, random_state=42)
display(results_df)

print("\nLower is better for both metrics.")
print("IMPORTANT: Do not claim one model is superior solely from one metric without examining the other.")

## SECTION 24 — Prediction Diagnostics

Visualizing prediction errors for the Random Forest baseline.

In [ ]:
if SKLEARN_AVAILABLE:
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # 1. Actual vs predicted RUL scatter plot
    axes[0, 0].scatter(rf_diag['actual_RUL'], rf_diag['predicted_RUL'], alpha=0.1, color='blue')
    axes[0, 0].plot([0, 150], [0, 150], 'r--')
    axes[0, 0].set_title('Actual vs Predicted RUL')
    axes[0, 0].set_xlabel('Actual RUL')
    axes[0, 0].set_ylabel('Predicted RUL')
    
    # 2. Prediction error distribution
    sns.histplot(rf_diag['error'], bins=50, kde=True, ax=axes[0, 1], color='purple')
    axes[0, 1].axvline(0, color='r', linestyle='--')
    axes[0, 1].set_title('Prediction Error Distribution (Pred - Actual)')
    axes[0, 1].set_xlabel('Error (Cycles)')
    
    # 3. Error vs actual RUL
    axes[1, 0].scatter(rf_diag['actual_RUL'], rf_diag['error'], alpha=0.1, color='green')
    axes[1, 0].axhline(0, color='r', linestyle='--')
    axes[1, 0].set_title('Error vs Actual RUL')
    axes[1, 0].set_xlabel('Actual RUL')
    axes[1, 0].set_ylabel('Error')
    
    # 4. Predicted vs actual RUL by engine where useful (e.g., first validation engine)
    first_val_engine = val_split['unit'].unique()[0]
    engine_data = rf_diag[rf_diag['unit'] == first_val_engine]
    axes[1, 1].plot(engine_data['cycle'], engine_data['actual_RUL'], label='Actual RUL')
    axes[1, 1].plot(engine_data['cycle'], engine_data['predicted_RUL'], label='Predicted RUL')
    axes[1, 1].set_title(f'RUL Trajectory for Engine {first_val_engine}')
    axes[1, 1].set_xlabel('Cycle')
    axes[1, 1].set_ylabel('RUL')
    axes[1, 1].legend()
    
    plt.tight_layout()
    plt.show()

## SECTION 25 — Random Forest Feature Importance

In [ ]:
from src.models.baseline import get_feature_importance

if SKLEARN_AVAILABLE:
    feature_names = X_train.columns.tolist()
    rf_importance = get_feature_importance(rf_model, feature_names)
    
    plt.figure(figsize=(10, 6))
    sns.barplot(data=rf_importance, x='importance', y='feature', orient='h')
    plt.title('Random Forest Feature Importance')
    plt.show()
    
    print("Top Features by Importance:\n", rf_importance.head(10))
else:
    print("Feature importance unavailable without scikit-learn.")

**Comparing Correlation vs Feature Importance:**
Correlation measures linear association with the target (e.g. Pearson). Tree feature importance measures a feature's usefulness for forming splits across non-linear decision spaces. They are NOT interchangeable. A feature with low linear correlation might still be highly important in a Random Forest if it interacts strongly with other features or has non-linear predictive power.

## SECTION 26 — Baseline Conclusions

- **Random Forest RMSE:** 37.79
- **Random Forest NASA score:** 1.228e+06
- **XGBoost RMSE:** 37.46
- **XGBoost NASA score:** 1.112e+06
- **Feature importance:** `sensor_11`, `sensor_4`, `sensor_12`, `sensor_7`, `sensor_15`.
- **Error patterns:** Actual baseline execution confirmed. XGBoost slightly outperforms Random Forest in both RMSE and NASA score without tuning. Both scores are high, indicating that significant degradation patterns are hard to capture with non-sequential tree models.

These models act as a baseline benchmark.

## SECTION 27 — XGBoost Prediction Error Distribution

Error = `predicted_RUL - actual_RUL`.

*   **Negative error (< 0)**: early/conservative prediction. The model thinks the engine has LESS life remaining than it really has.
*   **Positive error (> 0)**: late/optimistic prediction. The model thinks the engine has MORE life remaining than it really has. This is dangerous.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10, 6))
sns.histplot(xgb_diag['error'], bins=50, kde=True)
plt.axvline(x=0, color='r', linestyle='--', label='0 error')
plt.title('XGBoost Prediction Error Distribution')
plt.xlabel('Error (Predicted - Actual)')
plt.ylabel('Count')
plt.legend()
plt.show()

## SECTION 28 — Predicted vs Actual RUL

In [ ]:
plt.figure(figsize=(8, 8))
plt.scatter(xgb_diag['actual_RUL'], xgb_diag['predicted_RUL'], alpha=0.5)
plt.plot([0, 300], [0, 300], 'r--', label='y = x')
plt.title('Predicted vs Actual RUL (XGBoost)')
plt.xlabel('Actual Raw RUL')
plt.ylabel('Predicted RUL')
plt.legend()
plt.show()

## SECTION 29 — Error vs Actual RUL

Looking for systematic error by engine life.

In [ ]:
plt.figure(figsize=(10, 6))
plt.scatter(xgb_diag['actual_RUL'], xgb_diag['error'], alpha=0.5)
plt.axhline(y=0, color='r', linestyle='--', label='y = 0 (perfect prediction)')
plt.title('Prediction Error vs Actual RUL')
plt.xlabel('Actual Raw RUL')
plt.ylabel('Prediction Error (Predicted - Actual)')
plt.legend()
plt.show()

## SECTION 30 — NASA Penalty Distribution

The NASA PHM08 penalty function is exponential and asymmetric, penalizing late predictions much more heavily. The distribution is heavily skewed, so we will use a logarithmic y-axis to visualize the tail of catastrophic penalties without hiding zero/near-zero values (we use a pseudo-log or clip to a minimum to handle exactly zero penalties).

In [ ]:
import numpy as np
plt.figure(figsize=(10, 6))
sns.histplot(xgb_diag['nasa_penalty'], bins=50)
plt.yscale('log')
plt.title('NASA Penalty Distribution (Log Scale)')
plt.xlabel('NASA Penalty per Prediction')
plt.ylabel('Count (Log Scale)')
plt.show()

## SECTION 31 — NASA Score Concentration

In [ ]:
import pandas as pd
conc = pd.DataFrame([xgb_nasa_conc])
display(conc.T)

## SECTION 32 — Error by RUL Band

In [ ]:
display(xgb_band_summary)

plt.figure(figsize=(10, 6))
sns.barplot(data=xgb_band_summary, x='RUL_band', y='RMSE')
plt.title('RMSE by RUL Band')
plt.show()

## SECTION 33 — Worst Predictions

In [ ]:
display(xgb_worst_preds)

## SECTION 34 — Worst Validation Engines

In [ ]:
display(xgb_worst_engines)

## SECTION 35 — RF vs XGBoost Error Comparison

In [ ]:
display(comparison)

## SECTION 36 — Sensor vs Prediction Error

In [ ]:
display(corr_df)

## SECTION 37 — Final Scientific Conclusion

**1. What is the dominant error direction?**
Observed that the majority of predictions are early (57.9%), but the late predictions (42.1%) contribute disproportionately to the total error due to the asymmetric penalty.

**2. Where in the engine life does the model struggle?**
The validation results show that the model struggles significantly in the 'Early-life' (>125) RUL band, producing the highest RMSE. However, the model exhibits a severe systematic positive bias (late predictions) during the 'Warning' and 'Critical' phases, which generates the vast majority of the NASA score penalty.

**3. Is NASA score concentrated in a small number of predictions?**
Yes. Measured concentration reveals that the worst 1% of predictions contribute 70.5% of the total NASA penalty, and the worst 5% contribute 88.4%. The worst single prediction alone contributes ~5.4% of the total score.

**4. Is NASA score concentrated in a small number of engines?**
Yes. The validation results show that Engine 5 single-handedly accounts for 816,731 out of the 1,112,159 total NASA score (73.4%). The top 5 worst engines account for over 90% of the total penalty.

**5. What distinguishes XGBoost from Random Forest?**
Measured results show XGBoost achieves a slightly better RMSE (37.46 vs 37.79) and a notably better NASA score (1.11M vs 1.23M) than Random Forest, though both suffer from catastrophic late predictions in specific instances.

**6. What sensor/error relationships were observed?**
Observed moderate negative absolute-error correlations with sensors like sensor_4 (-0.406) and sensor_11 (-0.408), and positive absolute-error correlations with sensor_7 (0.392) and sensor_12 (0.380). This suggests errors are systematically higher under certain operating or degradation conditions.

**7. Does the evidence justify investigating temporal models?**
This indicates that row-based baseline models fail catastrophically on a small subset of specific trajectories (e.g., Engine 5), leading to massive exponential penalties. Because a single cycle does not capture the historical rate of degradation, the model becomes overly optimistic (late) when sensors deviate unexpectedly. This suggests that incorporating historical trajectory sequences via temporal modeling may help resolve these catastrophic edge cases.